# Projet OC P7 — Modélisation

Trois approches : modèle simple (TF-IDF), modèle avancé (CNN-LSTM + embeddings), BERT (DistilBERT).
Split train/validation/test : 60/20/20. Tracking MLflow.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import mlflow

mlflow.set_tracking_uri('file://../mlruns')
mlflow.set_experiment('air-paradis-sentiment')

In [ ]:
df = pd.read_csv('../data/processed/cleaned_dataset_sentiment.csv')
df.head()

## A. Modèles simples — TF-IDF

In [ ]:
X = df['cleaned_text']
y = df['target']

vectorizer = TfidfVectorizer(max_features=5000)
X_vec = vectorizer.fit_transform(X)

X_temp, X_test, y_temp, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print('Train:', y_train.value_counts().to_dict())
print('Val:', y_val.value_counts().to_dict())
print('Test:', y_test.value_counts().to_dict())

### Régression logistique

In [ ]:
with mlflow.start_run(run_name='tfidf_logistic'):
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_val_pred = model.predict(X_val)
    y_pred = model.predict(X_test)
    val_acc = accuracy_score(y_val, y_val_pred)
    test_acc = accuracy_score(y_test, y_pred)
    print(f'Validation accuracy: {val_acc:.4f}')
    print(f'Test accuracy: {test_acc:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Négatif', 'Positif']))
    mlflow.log_metric('val_accuracy', val_acc)
    mlflow.log_metric('test_accuracy', test_acc)
    mlflow.sklearn.log_model(model, 'model')

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Négatif', 'Positif']).plot()
plt.title('Logistic Regression — Matrice de confusion (test)')
plt.show()

### Random Forest

In [ ]:
with mlflow.start_run(run_name='tfidf_random_forest'):
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_val_pred = rf.predict(X_val)
    y_pred_rf = rf.predict(X_test)
    val_acc = accuracy_score(y_val, y_val_pred)
    test_acc = accuracy_score(y_test, y_pred_rf)
    print(f'Validation accuracy: {val_acc:.4f}')
    print(f'Test accuracy: {test_acc:.4f}')
    print(classification_report(y_test, y_pred_rf, target_names=['Négatif', 'Positif']))
    mlflow.log_metric('val_accuracy', val_acc)
    mlflow.log_metric('test_accuracy', test_acc)
    mlflow.sklearn.log_model(rf, 'model')

cm = confusion_matrix(y_test, y_pred_rf)
ConfusionMatrixDisplay(cm, display_labels=['Négatif', 'Positif']).plot()
plt.title('Random Forest — Matrice de confusion (test)')
plt.show()

## B. Modèles avancés — CNN-LSTM + embeddings + BERT

Pipeline complet via script (Word2Vec, GloVe, DistilBERT).

In [ ]:
from training.pipeline import run_pipeline
results = run_pipeline(sample_size=50000, skip_bert=False)
pd.DataFrame(results)

## C. Comparaison des modèles

In [ ]:
import json
from pathlib import Path
results_path = Path('../models/production/model_comparison.json')
if results_path.exists():
    comparison = pd.DataFrame(json.loads(results_path.read_text()))
    comparison[['model', 'val_accuracy', 'val_f1', 'test_accuracy', 'test_f1']]
else:
    print('Lancez le pipeline pour générer model_comparison.json')